# OpenPlaque — Series 6 Native All-Component Gap Analysis v1

This experiment asks whether any **left-associated part** of the native Series-6 CAS-Net coronary segmentation approaches the native aorta closely but stops short of contact.

It reuses the completed Series-6 CAS-Net prediction and native Series-6 aorta. No model inference, TotalSegmentator run, threshold relaxation, geodesic bridge, or synthetic vessel is performed.

Series 7 is included only as a reference comparison.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Reuse/cache controls — immediately after Drive mount
DRIVE_ROOT = "/content/drive/MyDrive/OpenPlaque"
DICOM_ROOT = "/content/drive/MyDrive/CCTA/DICOM/3221"
OUT = f"{DRIVE_ROOT}/Series6_Native_All_Component_Gap_Analysis_v1"

SERIES6_PRED = f"{DRIVE_ROOT}/Series6_Left_Coronary_Origin_Validation_v1/external_model_predictions/series6_alternate.nii.gz"
SERIES7_PRED = f"{DRIVE_ROOT}/Series6_Left_Coronary_Origin_Validation_v1/external_model_predictions/series7_reference.nii.gz"
SERIES6_AORTA = f"{DRIVE_ROOT}/Series6_Native_Ostium_Topology_v1/native_aorta/series6_aorta.nii.gz"

print("Output:", OUT)
print("Series 6 prediction:", SERIES6_PRED)
print("Series 7 prediction:", SERIES7_PRED)
print("Series 6 native aorta:", SERIES6_AORTA)

In [ ]:
import shutil, sys, subprocess, json
from pathlib import Path

OPENPLAQUE_PIN = "b425a6c026b3560ca4a17cbb2f8de7f36aa7bf44"
OPENPLAQUE_BRANCH = "series6-native-all-component-gap-analysis-from-main"

if Path("/content/OpenPlaque").exists():
    shutil.rmtree("/content/OpenPlaque")

!git clone -q --branch {OPENPLAQUE_BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {OPENPLAQUE_PIN}

%pip install -q pylibjpeg pylibjpeg-libjpeg
%pip install -q /content/OpenPlaque

for name in list(sys.modules):
    if name == "openplaque" or name.startswith("openplaque."):
        del sys.modules[name]

print("OpenPlaque pin:", subprocess.check_output(["git","-C","/content/OpenPlaque","rev-parse","HEAD"], text=True).strip())

In [ ]:
# Synthetic tests before science
from openplaque.series6_native_all_component_gap_analysis_v1 import synthetic_self_test
print(synthetic_self_test())
!cd /content/OpenPlaque && pytest -q tests/test_series6_native_all_component_gap_analysis_v1.py

In [ ]:
from openplaque.series6_native_all_component_gap_analysis_v1 import prepare

for p in [SERIES6_PRED, SERIES7_PRED, SERIES6_AORTA]:
    if not Path(p).is_file() or Path(p).stat().st_size == 0:
        raise FileNotFoundError(f"Required cached input missing: {p}")

prep = prepare(
    drive_root=DRIVE_ROOT,
    dicom_root=DICOM_ROOT,
    output_dir=OUT,
)

print("Series 6:", prep["series6_metadata"])
print("Root registration:")
print(json.dumps(prep["root_registration"], indent=2))
print("Cached inputs verified.")

In [ ]:
from openplaque.series6_native_all_component_gap_analysis_v1 import analyze

summary = analyze(
    drive_root=DRIVE_ROOT,
    dicom_root=DICOM_ROOT,
    output_dir=OUT,
)

print(json.dumps(summary["decision"], indent=2))
print("\nStatus:", summary["status"])

In [ ]:
# Quantitative review
import pandas as pd
from IPython.display import display

gaps = pd.read_csv(Path(OUT)/"path_associated_component_gaps.csv")
print("Series 6 left-associated subsets, sorted by native aortic gap:")
display(
    gaps[(gaps.series=="series6") & gaps.path.isin(["LAD","C6","C7"])]
    .sort_values(["gap_distance_mm","associated_voxels"], ascending=[True,False])
)

print("Series 6 RCA control subsets:")
display(gaps[(gaps.series=="series6") & (gaps.path=="RCA")].sort_values("gap_distance_mm"))

print("Series 7 comparison:")
display(gaps[gaps.series=="series7"].sort_values(["path","gap_distance_mm"]))

In [ ]:
# Automatic QC
from IPython.display import Image, display, HTML

display(Image(filename=str(Path(OUT)/"01_series6_left_gap_summary.png"), width=900))

for p in sorted(Path(OUT).glob("QC_series6_*_gap_planes.png")):
    print(p.name)
    display(Image(filename=str(p), width=1000))
    hp = Path(str(p).replace("_planes.png","_hu_profile.png"))
    if hp.exists():
        display(Image(filename=str(hp), width=750))

report = Path(OUT)/"OPENPLAQUE_SERIES6_NATIVE_ALL_COMPONENT_GAP_ANALYSIS_V1_REPORT.html"
display(HTML(report.read_text()))

In [ ]:
# Verify deliverables
expected = [
    "run_state.json",
    "summary.json",
    "decision.json",
    "preparation.json",
    "input_provenance.json",
    "path_associated_component_gaps.csv",
    "01_series6_left_gap_summary.png",
    "OPENPLAQUE_SERIES6_NATIVE_ALL_COMPONENT_GAP_ANALYSIS_V1_REPORT.html",
    "OPENPLAQUE_SERIES6_NATIVE_ALL_COMPONENT_GAP_ANALYSIS_V1_RESULTS.zip",
]
missing=[x for x in expected if not (Path(OUT)/x).exists()]
if missing:
    raise RuntimeError("Missing outputs: "+str(missing))
print("COMPLETE")
for x in expected:
    print(Path(OUT)/x)